# Task 2 Demo: Animal Verification Pipeline

This notebook demonstrates Task 2 pipeline usage (NER + image classification) and edge cases.

- Uses `pipeline.AnimalVerificationPipeline`
- Test cases include matching and non-matching pairs, missing text/image, and invalid images.

In [5]:
from pipeline import AnimalVerificationPipeline
from pathlib import Path

pipeline = AnimalVerificationPipeline(
    ner_model_path="./models/ner_model",
    image_classifier_path="./models/image_classifier",
    ner_confidence_threshold=0.3, 
    image_confidence_threshold=0.2, 
    use_fuzzy_matching=True
)
print('Pipeline initialized with trained models')

Initializing NER model...
✓ CUDA is available. Device: NVIDIA GeForce RTX 5070


d:\Work\test_tasks\.venv\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForTokenClassification: ['cls.predictions.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint 

Model loaded from ./models/ner_model
  Loaded from ./models/ner_model
Initializing Image Classification model...
✓ CUDA is available. Device: NVIDIA GeForce RTX 5070
Model loaded from ./models/image_classifier
  Loaded from ./models/image_classifier
Pipeline initialized with trained models


In [6]:
# Example test cases (these paths may need valid images in the folder or your own dataset).
positive = {'text': 'There is a dog in the picture', 'image_path': 'animals/dog/OIP-_6rwQJccAbxntDlKJMo3xAHaKM.jpeg'}
negative = {'text': 'There is a cat in the picture', 'image_path': 'animals/dog/OIP-_6rwQJccAbxntDlKJMo3xAHaKM.jpeg'}
edge_empty_text = {'text': '', 'image_path': 'animals/dog/OIP-_6rwQJccAbxntDlKJMo3xAHaKM.jpeg'}
edge_missing_image = {'text': 'There is a dog', 'image_path': 'animals/nonexistent.jpg'}

for case_name, case in [('positive', positive), ('negative', negative), ('empty_text', edge_empty_text), ('missing_image', edge_missing_image)]:
    result = pipeline.verify(case['text'], case['image_path'], return_details=True)
    print('---', case_name, '---')
    print(result)

--- positive ---
{'match': True, 'confidence': 0.9998424053192139, 'reason': 'Match found', 'details': {'text': 'There is a dog in the picture', 'image_path': 'animals/dog/OIP-_6rwQJccAbxntDlKJMo3xAHaKM.jpeg', 'extracted_animals': ['dog'], 'predicted_animal': 'dog', 'image_confidence': 0.9998424053192139, 'top_5_predictions': [{'class': 'dog', 'class_id': 4, 'confidence': 0.9998424053192139}, {'class': 'squirrel', 'class_id': 9, 'confidence': 8.606653136666864e-05}, {'class': 'sheep', 'class_id': 7, 'confidence': 3.112756530754268e-05}, {'class': 'horse', 'class_id': 6, 'confidence': 2.1963014660286717e-05}, {'class': 'spider', 'class_id': 8, 'confidence': 7.1944791670830455e-06}]}}
--- negative ---
{'match': False, 'confidence': 0.9998424053192139, 'reason': 'No matching animal in text', 'details': {'text': 'There is a cat in the picture', 'image_path': 'animals/dog/OIP-_6rwQJccAbxntDlKJMo3xAHaKM.jpeg', 'extracted_animals': ['cat'], 'predicted_animal': 'dog', 'image_confidence': 0.999

## Edge Cases Covered
- Empty text input -> `match=False`, `error='Empty text input'`
- Missing image file -> `match=False`, `error='Image not found'`
- Low-confidence prediction, no extracted entities, fuzzy match variations included.

## Notes
- Adjust `animals` folder paths to your dataset for real results.
- The pipeline uses ImageNet-pretrained model by default if no `image_classifier_path` is given.